In [1]:
# ============================================================
# H2: LABELING — UPDATED (4 Rules, Aggressive Philosophy)
#
# Rule 1: Seller Buyback < 30 days
#         Liu et al. (2023) arxiv:2305.01543
#
# Rule 2: Identity Trade (Self-Trade)
#         La Morgia et al. (2023), Oh (2024)
#
# Rule 3: Multi-hop Cycle < 7 days
#         Oh (2024), Von Wachter et al. (2022)
#
# Rule 4: High Transaction Count per Wallet Pair
#         Niu et al. (ACM WWW 2024)
#
# Philosophy: Aggressive — prefer FP over FN
# ============================================================

import sqlite3
import pandas as pd
import numpy as np

# Load raw data (before feature engineering)
# Kita perlu raw data karena labeling harus dari awal
conn = sqlite3.connect('../../dataset/nfts.sqlite/nfts.sqlite')

print("Loading raw transfers (June - August 2021)...")

JUNE_1 = 1622505600
SEPT_1 = 1630454400

transfers_raw = pd.read_sql_query(f"""
    SELECT
        transaction_hash,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    WHERE timestamp >= {JUNE_1}
      AND timestamp <  {SEPT_1}
""", conn)

print(f"Loaded: {len(transfers_raw):,} rows")

# Fix types
transfers_raw['timestamp_dt'] = pd.to_datetime(
    transfers_raw['timestamp'], unit='s'
)
transfers_raw['transaction_value'] = pd.to_numeric(
    transfers_raw['transaction_value'], errors='coerce'
).fillna(0.0)

# Filter sales only (value > 0)
sales = transfers_raw[
    transfers_raw['transaction_value'] > 0
].copy()
print(f"Sales only (value > 0): {len(sales):,}")

# Remove burn addresses
BURN = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
burn_lower = {a.lower() for a in BURN}

sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)

print(f"After burn filter: {len(sales_clean):,}")

# Sort by timestamp — critical for temporal rules
sales_clean = sales_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

# Initialize
sales_clean['is_wash_trading'] = 0
wash_hashes = set()

# Constants
MAX_SECS_BUYBACK  = 30 * 24 * 3600  # Rule 1: 30 days
MAX_SECS_CYCLE    = 7  * 24 * 3600  # Rule 3: 7 days
MAX_HOPS          = 10              # Rule 3: max hops
PAIR_TX_THRESHOLD = 3               # Rule 4: min tx count

grouped      = sales_clean.groupby(['nft_address', 'token_id'])
total_groups = len(grouped)

print(f"\nLabeling rules:")
print(f"  Rule 1 (Liu 2023)      : Seller buyback < 30 days")
print(f"  Rule 2 (La Morgia 2023): Identity trade (self-trade)")
print(f"  Rule 3 (Oh 2024)       : Multi-hop cycle < 7 days")
print(f"  Rule 4 (Niu 2024)      : High tx count per wallet pair >= {PAIR_TX_THRESHOLD}")
print(f"\nUnique NFT tokens: {total_groups:,}")
print(f"Processing Rules 1, 2, 3...")

r1_count = 0
r2_count = 0
r3_count = 0
processed = 0

for (nft_addr, token_id), group in grouped:

    if len(group) < 1:
        processed += 1
        continue

    group = group.sort_values(
        'timestamp'
    ).reset_index(drop=True)
    n = len(group)

    for i in range(n):
        from_i = group.loc[i, 'from_address'].lower()
        to_i   = group.loc[i, 'to_address'].lower()
        hash_i = group.loc[i, 'transaction_hash']
        ts_i   = group.loc[i, 'timestamp']

        # ------------------------------------------------
        # RULE 2: Identity Trade
        # La Morgia et al. (2023), Oh (2024)
        # Seller address == Buyer address
        # ------------------------------------------------
        if from_i == to_i:
            wash_hashes.add(hash_i)
            r2_count += 1
            continue  # no need to check further for this tx

        # Chain tracking for Rule 3
        chain_hashes = [hash_i]
        chain_end    = to_i

        for j in range(i + 1, n):
            ts_j      = group.loc[j, 'timestamp']
            diff_secs = ts_j - ts_i

            # ----------------------------------------
            # RULE 1: Seller Buyback < 30 days
            # Liu et al. (2023)
            # ----------------------------------------
            if diff_secs > MAX_SECS_BUYBACK:
                break

            from_j = group.loc[j, 'from_address'].lower()
            to_j   = group.loc[j, 'to_address'].lower()
            hash_j = group.loc[j, 'transaction_hash']

            # A sells → ... → A buys back
            if from_i == to_j:
                wash_hashes.add(hash_i)
                wash_hashes.add(hash_j)
                r1_count += 1

            # ----------------------------------------
            # RULE 3: Multi-hop Cycle < 7 days
            # Oh (2024), Von Wachter et al. (2022)
            # A→B→C→...→A within 7 days
            # ----------------------------------------
            if diff_secs <= MAX_SECS_CYCLE:
                if (from_j == chain_end and
                        len(chain_hashes) < MAX_HOPS):
                    chain_hashes.append(hash_j)
                    chain_end = to_j

                    if to_j == from_i:
                        for h in chain_hashes:
                            wash_hashes.add(h)
                        r3_count += 1
                        # Reset chain
                        chain_hashes = [hash_i]
                        chain_end    = to_i

    processed += 1
    if processed % 100000 == 0:
        print(f"  Progress: {processed:,} / "
              f"{total_groups:,} "
              f"({processed/total_groups:.1%})")

print(f"\nRule 1 (seller buyback)  : {r1_count:,} pairs")
print(f"Rule 2 (identity trade)  : {r2_count:,} transactions")
print(f"Rule 3 (multi-hop cycle) : {r3_count:,} cycles")
print(f"Total hashes so far      : {len(wash_hashes):,}")

# ============================================================
# RULE 4: High Transaction Count per Wallet Pair
# Niu et al. (ACM WWW 2024)
# Wallet pair (A, B) that traded the same NFT
# more than threshold times
# ============================================================
print(f"\nProcessing Rule 4 (High tx count per wallet pair)...")

r4_count     = 0
before_rule4 = len(wash_hashes)

# Count transactions per (from, to, nft_address, token_id)
pair_counts = sales_clean.groupby([
    'from_address', 'to_address',
    'nft_address', 'token_id'
]).size().reset_index(name='tx_count')

# Flag pairs exceeding threshold
suspicious_pairs = pair_counts[
    pair_counts['tx_count'] >= PAIR_TX_THRESHOLD
]

print(f"  Suspicious wallet pairs: {len(suspicious_pairs):,}")

for _, row in suspicious_pairs.iterrows():
    mask = (
        (sales_clean['from_address'] == row['from_address']) &
        (sales_clean['to_address']   == row['to_address']) &
        (sales_clean['nft_address']  == row['nft_address']) &
        (sales_clean['token_id']     == row['token_id'])
    )
    new_hashes = set(
        sales_clean[mask]['transaction_hash'].tolist()
    ) - wash_hashes
    wash_hashes.update(new_hashes)
    r4_count += len(new_hashes)

print(f"  New hashes from Rule 4 : {r4_count:,}")

# ============================================================
# APPLY LABELS
# ============================================================
sales_clean.loc[
    sales_clean['transaction_hash'].isin(wash_hashes),
    'is_wash_trading'
] = 1

# ============================================================
# RESULTS
# ============================================================
total     = len(sales_clean)
wt_count  = int(sales_clean['is_wash_trading'].sum())
norm_count = total - wt_count
ratio     = wt_count / norm_count

print(f"\n{'='*55}")
print(f"LABELING RESULTS (4 Rules)")
print(f"{'='*55}")
print(f"Total sales      : {total:,}")
print(f"Wash trading (1) : {wt_count:,} ({wt_count/total:.3%})")
print(f"Normal (0)       : {norm_count:,}")
print(f"Ratio            : 1 : {norm_count//wt_count}")
print(f"\nBreakdown:")
print(f"  Rule 1 (seller buyback)  : {r1_count:,} pairs")
print(f"  Rule 2 (identity trade)  : {r2_count:,} tx")
print(f"  Rule 3 (multi-hop cycle) : {r3_count:,} cycles")
print(f"  Rule 4 (high pair tx)    : {r4_count:,} new tx")

# Sanity check
wt_df      = sales_clean[sales_clean['is_wash_trading'] == 1]
burn_in_wt = wt_df[
    wt_df['to_address'].str.lower().isin(burn_lower)
]
print(f"\nSanity check:")
print(f"  Burn addresses in labels: {len(burn_in_wt)}")
print(f"  (Expected: 0)")

# Check if ratio is within 1:50
if norm_count // wt_count <= 50:
    print(f"\n✅ Ratio {norm_count//wt_count}:1 — within 1:50 limit!")
else:
    print(f"\n⚠️ Ratio {norm_count//wt_count}:1 — exceeds 1:50!")
    print(f"   Need to undersample normal class")
    print(f"   Target normal: {wt_count * 50:,}")
    print(f"   Need to remove: {norm_count - wt_count*50:,}")

Loading raw transfers (June - August 2021)...
Loaded: 2,534,558 rows
Sales only (value > 0): 1,581,258
After burn filter: 1,562,654

Labeling rules:
  Rule 1 (Liu 2023)      : Seller buyback < 30 days
  Rule 2 (La Morgia 2023): Identity trade (self-trade)
  Rule 3 (Oh 2024)       : Multi-hop cycle < 7 days
  Rule 4 (Niu 2024)      : High tx count per wallet pair >= 3

Unique NFT tokens: 1,164,974
Processing Rules 1, 2, 3...
  Progress: 100,000 / 1,164,974 (8.6%)
  Progress: 200,000 / 1,164,974 (17.2%)
  Progress: 300,000 / 1,164,974 (25.8%)
  Progress: 400,000 / 1,164,974 (34.3%)
  Progress: 500,000 / 1,164,974 (42.9%)
  Progress: 600,000 / 1,164,974 (51.5%)
  Progress: 700,000 / 1,164,974 (60.1%)
  Progress: 800,000 / 1,164,974 (68.7%)
  Progress: 900,000 / 1,164,974 (77.3%)
  Progress: 1,000,000 / 1,164,974 (85.8%)
  Progress: 1,100,000 / 1,164,974 (94.4%)

Rule 1 (seller buyback)  : 4,971 pairs
Rule 2 (identity trade)  : 57 transactions
Rule 3 (multi-hop cycle) : 1,410 cycles
Total 

In [2]:
print("="*60)
print("WALLET ACTIVITY ANALYSIS")
print("="*60)

# Semua wallet muncul sebagai sender atau receiver

sender_counts = sales_clean.groupby(
    'from_address'
).size()

receiver_counts = sales_clean.groupby(
    'to_address'
).size()

wallet_activity = sender_counts.add(
    receiver_counts,
    fill_value=0
)

wallet_activity = wallet_activity.astype(int)

print("\nWallet statistics:")
print(wallet_activity.describe())

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(
        f"P{p}: "
        f"{wallet_activity.quantile(p/100):.0f} transactions"
    )

WALLET ACTIVITY ANALYSIS

Wallet statistics:
count    202631.000000
mean         15.423642
std         205.100468
min           1.000000
25%           1.000000
50%           2.000000
75%           8.000000
max       87316.000000
dtype: float64

Percentiles:
P50: 2 transactions
P75: 8 transactions
P90: 29 transactions
P95: 60 transactions
P99: 214 transactions


In [3]:
fraud_df = sales_clean[
    sales_clean['is_wash_trading'] == 1
]

fraud_wallets = set(
    fraud_df['from_address']
).union(
    set(fraud_df['to_address'])
)

fraud_wallet_activity = wallet_activity[
    wallet_activity.index.isin(fraud_wallets)
]

print("="*60)
print("FRAUD WALLET HISTORY")
print("="*60)

print(f"Fraud wallets: {len(fraud_wallet_activity):,}")

print(
    fraud_wallet_activity.describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{fraud_wallet_activity.quantile(p/100):.0f}"
    )

FRAUD WALLET HISTORY
Fraud wallets: 1,800
count    1800.000000
mean      130.136667
std       323.791386
min         1.000000
25%         5.000000
50%        27.000000
75%       114.000000
max      6608.000000
dtype: float64
P50: 27
P75: 114
P90: 355
P95: 593
P99: 1423


In [4]:
nft_chain_lengths = sales_clean.groupby(
    ['nft_address','token_id']
).size()

print("="*60)
print("NFT CHAIN LENGTH")
print("="*60)

print(nft_chain_lengths.describe())

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{nft_chain_lengths.quantile(p/100):.0f}"
    )

NFT CHAIN LENGTH
count    1.164974e+06
mean     1.341364e+00
std      7.049694e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.100000e+01
dtype: float64
P50: 1
P75: 1
P90: 2
P95: 3
P99: 4


In [5]:
fraud_tx_per_wallet = pd.concat([
    fraud_df['from_address'],
    fraud_df['to_address']
]).value_counts()

print(
    fraud_tx_per_wallet.describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{fraud_tx_per_wallet.quantile(p/100):.0f}"
    )

count    1800.000000
mean       12.738889
std       140.299492
min         1.000000
25%         2.000000
50%         2.000000
75%         4.000000
max      4822.000000
Name: count, dtype: float64
P50: 2
P75: 4
P90: 13
P95: 26
P99: 99


In [7]:
fraud_wallets = set(
    fraud_df['from_address']
).union(
    set(fraud_df['to_address'])
)

# Ubah ke long format
wallet_events = pd.concat([
    sales_clean[['from_address', 'timestamp']]
        .rename(columns={'from_address':'wallet'}),
    sales_clean[['to_address', 'timestamp']]
        .rename(columns={'to_address':'wallet'})
])

wallet_events = wallet_events[
    wallet_events['wallet'].isin(fraud_wallets)
]

wallet_span = wallet_events.groupby(
    'wallet'
)['timestamp'].agg(['min','max'])

wallet_span['lifespan_days'] = (
    wallet_span['max'] -
    wallet_span['min']
) / 86400

print(wallet_span['lifespan_days'].describe())

for p in [50,75,90,95]:
    print(
        f"P{p}: "
        f"{wallet_span['lifespan_days'].quantile(p/100):.1f} days"
    )

count    1800.000000
mean       38.508853
std        33.142509
min         0.000000
25%         5.506916
50%        30.408258
75%        71.955943
max        91.907500
Name: lifespan_days, dtype: float64
P50: 30.4 days
P75: 72.0 days
P90: 87.1 days
P95: 90.4 days
